In [ ]:
import math
import re
import datasets
from datasets import load_dataset
import torch
import torch.nn as nn
from torch.nn import functional as F

# 1. DOWNLOAD E PREPARAÇÃO DOS DADOS (WIKIPÉDIA PT)

print('Baixando amostra da Wikipédia em Português...')
dataset = load_dataset('wikimedia/wikipedia', '20231101.pt', split='train[:10000]')
raw_text = '\n\n'.join(dataset['text'])


def clean_wiki_text(t):
  t = re.sub(r'http\S+', '', t)
  t = re.sub(r'==+.*?==+', '', t)
  t = re.sub(r'[\x91\x92\x93\x94\x96\x97\x9c\xa0\xad]', '', t)
  t = re.sub(r'[ \t]+', ' ', t)
  t = re.sub(r'\n+', '\n', t)
  return t


text = clean_wiki_text(raw_text)

# Cria o vocabulário e mapeamentos
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s if c in stoi]
decode = lambda l: ''.join([itos[i] for i in l])

# Transforma em Tensores PyTorch
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(
    f'Dados prontos! Vocabulário: {vocab_size} caracteres | Treino:'
    f' {len(train_data)} tokens'
)


# 2. ARQUITETURA DO TRANSFORMER

class Head(nn.Module):
  """Uma cabeça de Causal Self-Attention"""

  def __init__(self, head_size, n_embd, block_size, dropout):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer(
        'tril', torch.tril(torch.ones(block_size, block_size))
    )
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B, T, C = x.shape
    k = self.key(x)
    q = self.query(x)
    wei = q @ k.transpose(-2, -1) * (C ** -0.5)
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)
    v = self.value(x)
    out = wei @ v
    return out


class MultiHeadAttention(nn.Module):
  """Várias cabeças de Atenção rodando em paralelo"""

  def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
    super().__init__()
    self.heads = nn.ModuleList([
        Head(head_size, n_embd, block_size, dropout) for _ in range(num_heads)
    ])
    self.proj = nn.Linear(n_embd, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.dropout(self.proj(out))
    return out


class FeedForward(nn.Module):
  """Camada Linear + GELU + Projeção (MLP interna)"""

  def __init__(self, n_embd, dropout):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4 * n_embd),
        nn.GELU(),
        nn.Linear(4 * n_embd, n_embd),
        nn.Dropout(dropout),
    )

  def forward(self, x):
    return self.net(x)


class Block(nn.Module):
  """Bloco Transformer: Atenção + FeedForward com conexões residuais"""

  def __init__(self, n_embd, n_head, block_size, dropout):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size, n_embd, block_size, dropout)
    self.ffwd = FeedForward(n_embd, dropout)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x


class GPTLanguageModel(nn.Module):

  def __init__(
      self, vocab_size, n_embd, n_head, n_layer, block_size, dropout
  ):
    super().__init__()
    self.block_size = block_size
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(
        *[
            Block(n_embd, n_head, block_size, dropout)
            for _ in range(n_layer)
        ]
    )
    self.ln_f = nn.LayerNorm(n_embd)
    self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, idx, targets=None):
    B, T = idx.shape
    tok_emb = self.token_embedding_table(idx)
    pos_emb = self.position_embedding_table(
        torch.arange(T, device=idx.device)
    )
    x = tok_emb + pos_emb
    x = self.blocks(x)
    x = self.ln_f(x)
    logits = self.lm_head(x)

    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B * T, C)
      targets = targets.view(B * T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens, temperature=0.8):
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -self.block_size :]
      logits, _ = self(idx_cond)
      logits = logits[:, -1, :] / temperature
      probs = F.softmax(logits, dim=-1)
      idx_next = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx



# 3. FUNÇÕES DE TREINAMENTO E GERAÇÃO

def treinar_gpt(
    train_data,
    val_data,
    vocab_size,
    block_size=64,
    batch_size=32,
    n_embd=128,
    n_head=4,
    n_layer=4,
    dropout=0.1,
    max_iters=3000,
    eval_interval=500,
    lr=1e-3,
):
  device = 'cuda' if torch.cuda.is_available() else 'cpu'
  print(f'Treinando no dispositivo: {device.upper()}')

  def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in ix])
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

  model = GPTLanguageModel(
      vocab_size, n_embd, n_head, n_layer, block_size, dropout
  ).to(device)
  optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

  print(
      f'Total de parâmetros: {sum(p.numel() for p in model.parameters())/1e6:.2f}M'
  )

  for iter in range(max_iters):
    if iter % eval_interval == 0:
      model.eval()
      with torch.no_grad():
        xb, yb = get_batch('val')
        _, val_loss = model(xb, yb)
        print(f'Passo {iter:4d} | Perda de Validação: {val_loss.item():.4f}')
      model.train()

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

  print('\nTreinamento concluído com sucesso!')
  return model, device


def gerar_texto_gpt(
    model, device, stoi, itos, prompt, block_size=64, max_chars=400, temp=0.8
):
  model.eval()
  encoded = [stoi[c] for c in prompt if c in stoi]
  if not encoded:
    encoded = [0]
  context = torch.tensor([encoded], dtype=torch.long, device=device)
  with torch.no_grad():
    generated = model.generate(
        context, max_new_tokens=max_chars, temperature=temp
    )
  return ''.join([itos[i] for i in generated[0].tolist()])





Baixando amostra da Wikipédia em Português...
Dados prontos! Vocabulário: 2530 caracteres | Treino: 82672050 tokens


In [ ]:
"""modelo_treinado, dispositivo = treinar_gpt(
    train_data=train_data,
    val_data=val_data,
    vocab_size=vocab_size,
    block_size=64,
    max_iters=3000,
)

print('\n--- Resultado 1 ---')
print(
    gerar_texto_gpt(
        modelo_treinado,
        dispositivo,
        stoi,
        itos,
        prompt='A ciência é',
        max_chars=300,
        temp=0.7,
    )
)

print('\n--- Resultado 2 ---')
print(
    gerar_texto_gpt(
        modelo_treinado,
        dispositivo,
        stoi,
        itos,
        prompt='O Brasil possui',
        max_chars=300,
        temp=0.8,
    )
)"""

In [ ]:
print(
    gerar_texto_gpt(
        modelo_treinado,
        dispositivo,
        stoi,
        itos,
        prompt='Napoleão Bonaparte foi',
        max_chars=500,
        temp=0.5,
    )
)

Napoleão Bonaparte foi até proples de algumas como o mundo como o mande alguma proponesa e ainda a capital de disposição e é a pré-limina de estados de portuguesa. Estados a antigação e como uma população do mais entre se manholada a remografia de artigos da como ou assim que até a propriedade de compostas em cada dos antigos processos em infanteses de propassas da América de 1990 e algumas por 2000 9. Algumas de 2000, para a reformação de corrente de desenvolver proposição como o atribui e para a teoria de através d


In [ ]:
modelo_treinado_pro, dispositivo_pro = treinar_gpt(
    train_data=train_data,
    val_data=val_data,
    vocab_size=vocab_size,
    block_size=128,
    max_iters=15000,
)


Treinando no dispositivo: CUDA
Total de parâmetros: 1.46M
Passo    0 | Perda de Validação: 7.9467
Passo  500 | Perda de Validação: 2.3421
Passo 1000 | Perda de Validação: 2.0942
Passo 1500 | Perda de Validação: 1.7887
Passo 2000 | Perda de Validação: 1.7046
Passo 2500 | Perda de Validação: 1.6293
Passo 3000 | Perda de Validação: 1.6614
Passo 3500 | Perda de Validação: 1.5907
Passo 4000 | Perda de Validação: 1.4724
Passo 4500 | Perda de Validação: 1.6054
Passo 5000 | Perda de Validação: 1.6223
Passo 5500 | Perda de Validação: 1.5573
Passo 6000 | Perda de Validação: 1.4109
Passo 6500 | Perda de Validação: 1.4618
Passo 7000 | Perda de Validação: 1.4325
Passo 7500 | Perda de Validação: 1.3888
Passo 8000 | Perda de Validação: 1.4642
Passo 8500 | Perda de Validação: 1.4046
Passo 9000 | Perda de Validação: 1.2922
Passo 9500 | Perda de Validação: 1.4937
Passo 10000 | Perda de Validação: 1.4504
Passo 10500 | Perda de Validação: 1.4529
Passo 11000 | Perda de Validação: 1.3826
Passo 11500 | Perda

In [ ]:
print(
    gerar_texto_gpt(
        modelo_treinado_pro,
        dispositivo_pro,
        stoi,
        itos,
        prompt='Napoleão Bonaparte foi',
        max_chars=500,
        temp=0.8,
    )
)

Napoleão Bonaparte foi a Chinappe Mirama (m. 1934).
 1866
Ribeiro - Eduardo Mines do Candomblé Pobro, com o maior nome do Império Alvano do Rio António Dúlio de Salvador, do Património Mundial de São Paulo. Devido à maior de 34 de maio de 13 de julho de 1843, em 1878, após um seu país tem em Beatalha golpeza no metro dos diploma de banco cujo paulista artificial é feito do mundo, pesquisador de outros novos campos de digisão serviços. Nesses se além de uma pessoas para referências de altitude do grupo do carro de por


In [ ]:
print(
    gerar_texto_gpt(
        modelo_treinado_pro,
        dispositivo_pro,
        stoi,
        itos,
        prompt='Quando Napoleão Morreu?',
        max_chars=500,
        temp=0.8,
    )
)

Quando Napoleão Morreu?. Hajia em 1992, com crescente dos Denses Califórnicos Fernandesa francesa da Macífica (1938).
 1963 — Mineiro Maria Dia da Nita, acordo com três freguesias, desemplanos de Annabinação (m. 1985).
 1978 — França do Castelo e Suécia do Sul Russo e, em Michael de Ruís Salvenente, a fraca dos Meses, a região na Internet e por exemplos do Sul, e que depois iniciou a que compra em alcançamentos dois primeiros dois acordos de visão voltados espados. Os principais tipos de vila distância e foram continu
